CÉLULA 1: Instalações

In [5]:
!pip install -q streamlit openai-whisper easyocr transformers pydub accelerate pyngrok PyMuPDF python-docx duckduckgo-search pandas psutil bitsandbytes hf_transfer

Célula 2: Motor + Interface

In [6]:
%%writefile app_piloto.py
import streamlit as st
import time
import os
import shutil
import torch
import gc
import whisper
import easyocr
import psutil
import pandas as pd
from pydub import AudioSegment
import fitz
import docx
import requests
import json
from io import BytesIO
from transformers import pipeline, BitsAndBytesConfig
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")

# ─────────────────────────────────────────
# 0. ESTADOS DA APLICAÇÃO
# ─────────────────────────────────────────
if 'etapa' not in st.session_state: st.session_state.etapa = 0
if 'resumo' not in st.session_state: st.session_state.resumo = ""
if 'termos' not in st.session_state: st.session_state.termos = ""
if 'relatorio' not in st.session_state: st.session_state.relatorio = ""
if 'debug_juris' not in st.session_state: st.session_state.debug_juris = ""
if 'debug_resposta_bruta' not in st.session_state: st.session_state.debug_resposta_bruta = ""
if 'debug_prompt' not in st.session_state: st.session_state.debug_prompt = ""

# Estados da V2.2 (Petição Inicial Backward Design + Cache)
if 'texto_fatos' not in st.session_state: st.session_state.texto_fatos = ""
if 'texto_direito' not in st.session_state: st.session_state.texto_direito = ""
if 'texto_tutela' not in st.session_state: st.session_state.texto_tutela = ""
if 'texto_pedidos' not in st.session_state: st.session_state.texto_pedidos = ""
if 'doc_gerado' not in st.session_state: st.session_state.doc_gerado = None
if 'msg_status' not in st.session_state: st.session_state.msg_status = ""

# ─────────────────────────────────────────
# 1. FUNÇÕES DE PROCESSAMENTO E UTILIDADES
# ─────────────────────────────────────────

# NOVIDADE: Cache do Modelo (Carrega só 1 vez na memória RAM/VRAM)
@st.cache_resource
def carregar_modelo_llm():
    conf = BitsAndBytesConfig(load_in_8bit=True)
    return pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct", device_map="auto", model_kwargs={"quantization_config": conf})

def ler_documentos_texto(pasta_caso):
    conteudo = ""
    for arq in [f for f in os.listdir(pasta_caso) if f.endswith(('.txt', '.pdf', '.docx'))]:
        caminho = os.path.join(pasta_caso, arq)
        try:
            if arq.endswith('.txt'):
                with open(caminho, 'r', encoding='utf-8') as f: conteudo += f.read() + "\n"
            elif arq.endswith('.pdf'):
                doc = fitz.open(caminho)
                for pag in doc: conteudo += pag.get_text() + "\n"
                doc.close()
            elif arq.endswith('.docx'):
                doc = docx.Document(caminho)
                for p in doc.paragraphs: conteudo += p.text + "\n"
        except: pass
    return conteudo

def extrair_texto_audios(pasta_caso, m_whisper):
    arqs = [f for f in os.listdir(pasta_caso) if f.endswith(('.ogg', '.mp3', '.wav'))]
    if not arqs: return "Sem áudio."
    som = AudioSegment.empty()
    for f in sorted(arqs): som += AudioSegment.from_file(os.path.join(pasta_caso, f))
    som.export("temp.wav", format="wav")
    return m_whisper.transcribe("temp.wav", language="pt")["text"]

def extrair_texto_imagens(pasta_caso, leitor_ocr):
    arqs = [f for f in os.listdir(pasta_caso) if f.endswith(('.png', '.jpg', '.jpeg'))]
    texto = ""
    for arq in arqs:
        leitura = leitor_ocr.readtext(os.path.join(pasta_caso, arq), detail=0)
        texto += f"\n[PROVA VISUAL {arq}]: {' '.join(leitura)}"
    return texto

def gerar_docx_peticao(fatos, direito, tutela, pedidos):
    try:
        doc = docx.Document("modelo_peticao.docx")
    except Exception as e:
        return None, f"Erro: Faça o upload do arquivo 'modelo_peticao.docx'. Detalhe: {e}"

    substituicoes = {
        '{{FATOS}}': fatos,
        '{{DIREITO}}': direito,
        '{{TUTELA}}': tutela,
        '{{PEDIDOS}}': pedidos
    }

    # Motor de injeção que respeita quebras de linha (Enters)
    for para in doc.paragraphs:
        for tag, texto_novo in substituicoes.items():
            if tag in para.text:
                para.text = para.text.replace(tag, '')
                run = para.add_run()
                linhas = texto_novo.split('\n')
                for i, linha in enumerate(linhas):
                    run.add_text(linha)
                    if i < len(linhas) - 1:
                        run.add_break()

    file_stream = BytesIO()
    doc.save(file_stream)
    file_stream.seek(0)
    return file_stream, "Sucesso"

# ─────────────────────────────────────────
# 2. AGENTES IA & MOTOR DE BUSCA
# ─────────────────────────────────────────
def agente_investigador(audio, docs, prints, anotacoes, modelo):
    prompt = f"""Você é um perito jurídico sênior (Brasil).
Sua missão é extrair os fatos e gerar a string de busca.

DADOS BRUTOS:
Anotações: {anotacoes}
Docs: {docs[:4000]}
Áudio: {audio[:4000]}
Provas: {prints[:2000]}

FORMATO OBRIGATÓRIO DE RESPOSTA:
<RESUMO_LIMPO>
[Escreva o relato dos fatos de forma cronológica, clara e objetiva. EVITE ADJETIVOS desnecessários. Numere os parágrafos (1, 2, 3...).]
</RESUMO_LIMPO>
<TERMOS_BUSCA>
[Gere EXATAMENTE 3 teses jurídicas centrais focadas EXCLUSIVAMENTE NO MÉRITO (Direito Material).
REGRA: É expressamente PROIBIDO gerar termos de busca sobre questões processuais (como "justiça gratuita", "prioridade de tramitação" ou "competência"). Foque no dano, na conduta e na responsabilidade (Ex: "falha na prestação de serviço", "fraude bancária", "dano moral").
Coloque cada tese obrigatoriamente entre aspas duplas, separadas por espaço. NADA MAIS.]
</TERMOS_BUSCA>"""
    msg = [{"role": "user", "content": prompt}]
    saida = modelo(msg, max_new_tokens=400, do_sample=False, repetition_penalty=1.15)
    texto = saida[0]["generated_text"][-1]["content"] if isinstance(saida[0]["generated_text"], list) else saida[0]["generated_text"]
    resumo = texto.split("<RESUMO_LIMPO>")[1].split("</RESUMO_LIMPO>")[0].strip() if "<RESUMO_LIMPO>" in texto else "Falha no resumo."
    termos_brutos = texto.split("<TERMOS_BUSCA>")[1].split("</TERMOS_BUSCA>")[0].strip() if "<TERMOS_BUSCA>" in texto else "jurisprudencia"
    termos_limpos = termos_brutos.replace("\n", " ").strip()
    if len(termos_limpos) > 80: termos_limpos = termos_limpos[:80] + '"'
    return resumo, termos_limpos

def buscar_na_web(termos):
    API_KEY = os.environ.get("SERPER_API")
    url = "https://google.serper.dev/search"
    query_google = f'{termos} jurisprudencia -filetype:pdf (site:stj.jus.br OR site:tjrj.jus.br OR site:tjsp.jus.br OR site:jusbrasil.com.br)'
    payload = json.dumps({"q": query_google, "gl": "br", "hl": "pt-br", "num": 10})
    headers = {'X-API-KEY': API_KEY, 'Content-Type': 'application/json'}
    try:
        response = requests.request("POST", url, headers=headers, data=payload)
        if response.status_code == 403: return "⚠️ AVISO: Cota gratuita esgotada."
        resultados_json = response.json()
        if 'organic' in resultados_json:
            resultados = resultados_json['organic'][:10]
            return "\n".join([f"[{r.get('title', 'Sem Título')} - {r.get('link', '#')}]: {r.get('snippet', 'Sem resumo')}" for r in resultados])
        else: return "Nenhuma jurisprudência encontrada."
    except Exception as e: return f"Erro na pesquisa: {e}"

def agente_relator(resumo, juris, anotacoes, modelo):
    prompt = f"""Você é um Paralegal sênior. Ignore o lixo de internet. Extraia a tese jurídica.
REGRA HIPERLINK: Use o formato **[Nome](URL)**.
RESUMO: {resumo}
JURISPRUDÊNCIA: {juris}
ANOTAÇÕES: {anotacoes}

MOLDE ESTRITO:
### 1. Partes Envolvidas
- **Cliente:** [Nome]
- **Parte contrária:** [Nome]

### 2. Resumo dos Fatos
[Narrativa]

### 3. Fundamentação Jurídica e Jurisprudência
[Teses com hiperlinks]

### 4. Viabilidade e Próximos Passos
[Parecer]"""
    msg = [{"role": "user", "content": prompt}]
    saida = modelo(msg, max_new_tokens=1000, do_sample=True, temperature=0.3, repetition_penalty=1.15)
    texto_bruto = saida[0]["generated_text"][-1]["content"] if isinstance(saida[0]["generated_text"], list) else saida[0]["generated_text"]
    return texto_bruto.split("MOLDE ESTRITO:")[-1].strip() if "MOLDE ESTRITO:" in texto_bruto else texto_bruto, texto_bruto, prompt

def agente_redator_peticao(relatorio, modelo):
    prompt = f"""Você é um Advogado Sênior redigindo uma Petição Inicial.
Abaixo está o relatório de triagem do caso. Baseado EXCLUSIVAMENTE nele, redija as seções da peça obedecendo estritamente ao estilo solicitado.

REGRA DE FORMATAÇÃO ABSOLUTA E INQUEBRÁVEL:
Você DEVE OBRIGATORIAMENTE iniciar e terminar cada seção com as tags XML exatas fornecidas no molde (<FATOS> e </FATOS>, etc). O sistema de injeção depende dessas tags. Se você não usá-las, o documento falhará.

RELATÓRIO DO CASO:
{relatorio}

MOLDE OBRIGATÓRIO DE RESPOSTA:
<FATOS>
[Narrativa cronológica, clara e objetiva. Evite adjetivos desnecessários. Cada fato relevante em um parágrafo NUMERADO (1., 2., 3...).]
</FATOS>

<DIREITO>
[Divida em teses (ex: II.1, II.2). Citar dispositivos legais aplicáveis.
REGRA DE JURISPRUDÊNCIA: Não faça paráfrases. Transcreva exatamente o texto fornecido no relatório e cite a fonte/tribunal logo abaixo entre parênteses.]
</DIREITO>

<TUTELA>
[Demonstre a probabilidade do direito e o perigo de dano. Se não houver urgência, escreva apenas "Não há pedido de tutela provisória de urgência aplicável aos fatos narrados."]
</TUTELA>

<PEDIDOS>
[Liste os pedidos finais utilizando verbos no infinitivo.
REGRA VITAL: O primeiro e mais importante pedido deve ser o de MÉRITO (condenação, devolução de valores, indenização por danos, etc), extraído diretamente da dor narrada nos fatos. Depois, liste os pedidos padrões processuais (citação, justiça gratuita, honorários).]
</PEDIDOS>"""

    msg = [{"role": "user", "content": prompt}]
    saida = modelo(msg, max_new_tokens=1800, do_sample=True, temperature=0.3, repetition_penalty=1.15)

    texto_bruto = saida[0]["generated_text"][-1]["content"] if isinstance(saida[0]["generated_text"], list) else saida[0]["generated_text"]

    def extrair(tag_inicio, tag_fim, texto):
        if tag_inicio in texto and tag_fim in texto:
            return texto.split(tag_inicio)[1].split(tag_fim)[0].strip()
        return f"⚠️ FALHA DE TAG: A IA não usou a tag {tag_inicio}.\n\n--- VEJA O QUE ELA ESCREVEU ABAIXO ---\n\n{texto[:1000]}..."

    fatos = extrair("<FATOS>", "</FATOS>", texto_bruto)
    direito = extrair("<DIREITO>", "</DIREITO>", texto_bruto)
    tutela = extrair("<TUTELA>", "</TUTELA>", texto_bruto)

    return fatos, direito, tutela, extrair("<PEDIDOS>", "</PEDIDOS>", texto_bruto)

# ─────────────────────────────────────────
# 3. INTERFACE STREAMLIT
# ─────────────────────────────────────────
st.set_page_config(page_title="Paralegal Digital", layout="wide")
st.markdown("""<style>.stApp { background-color: #F5F2ED; } .relatorio { background: white; padding: 2rem; border-left: 5px solid #1A365D; border-radius: 8px; margin-bottom: 2rem; }</style>""", unsafe_allow_html=True)

st.title("⚖️ Paralegal Digital v2.2 (Produção)")

with st.sidebar:
    st.header("1. Upload e Configuração")
    modo_execucao = st.radio("Modo de Operação:", ["⚡ Automatizado", "🛑 Interativo"])
    arquivos = st.file_uploader("Suba áudios, PDFs, DOCX ou imagens", accept_multiple_files=True)
    anotacoes = st.text_area("Notas iniciais")

    if st.button("🚀 Iniciar Triagem", type="primary", use_container_width=True):
        if arquivos:
            st.session_state.etapa = 0
            shutil.rmtree("temp", ignore_errors=True); os.makedirs("temp", exist_ok=True)
            for a in arquivos:
                with open(f"temp/{a.name}", "wb") as f: f.write(a.getbuffer())

            with st.status("Analisando fatos...", expanded=True) as s:
                m_w = whisper.load_model("medium"); txt_audio = extrair_texto_audios("temp", m_w); del m_w; gc.collect(); torch.cuda.empty_cache()
                m_o = easyocr.Reader(['pt'], gpu=True); txt_prints = extrair_texto_imagens("temp", m_o); del m_o; gc.collect(); torch.cuda.empty_cache()
                txt_docs = ler_documentos_texto("temp")

                # Chamada via Cache
                m_l = carregar_modelo_llm()

                st.session_state.resumo, st.session_state.termos = agente_investigador(txt_audio, txt_docs, txt_prints, anotacoes, m_l)

                if "Automatizado" in modo_execucao:
                    st.session_state.debug_juris = buscar_na_web(st.session_state.termos)
                    st.session_state.relatorio, st.session_state.debug_resposta_bruta, st.session_state.debug_prompt = agente_relator(st.session_state.resumo, st.session_state.debug_juris, anotacoes, m_l)
                    st.session_state.etapa = 2
                else:
                    st.session_state.etapa = 1

if st.session_state.etapa == 1:
    st.warning("🛑 **Modo Interativo:** Revise os termos de busca gerados.")
    col1, col2 = st.columns([1, 1])
    with col1: st.info(st.session_state.resumo)
    with col2:
        termos_editados = st.text_input("Edite os termos:", value=st.session_state.termos)
        if st.button("Continuar e Gerar Relatório", type="primary"):
            st.session_state.termos = termos_editados
            with st.spinner("Consultando Google..."):
                st.session_state.debug_juris = buscar_na_web(termos_editados)
                m_l = carregar_modelo_llm()
                st.session_state.relatorio, st.session_state.debug_resposta_bruta, st.session_state.debug_prompt = agente_relator(st.session_state.resumo, st.session_state.debug_juris, anotacoes, m_l)
                st.session_state.etapa = 2
                st.rerun()

if st.session_state.etapa == 2:
    st.success("✅ Relatório de Triagem Gerado!")
    with st.container(border=True): st.markdown(st.session_state.relatorio)

    st.divider()
    st.markdown("### 📝 Passo 2: Geração de Petição Inicial")
    st.info("O Agente redigirá a peça focando OBRIGATORIAMENTE no mérito.")

    if st.button("✨ Rascunhar Petição Inicial", type="primary", use_container_width=True):
        with st.spinner("Construindo o esqueleto jurídico..."):
            m_l = carregar_modelo_llm()
            fatos, direito, tutela, pedidos = agente_redator_peticao(st.session_state.relatorio, m_l)
            st.session_state.texto_fatos = fatos
            st.session_state.texto_direito = direito
            st.session_state.texto_tutela = tutela
            st.session_state.texto_pedidos = pedidos
            st.session_state.doc_gerado = None # Reseta o doc antigo
            st.session_state.etapa = 3
            st.rerun()

if st.session_state.etapa == 3:
    st.markdown("### 👁️ Revisão Humana (Human-in-the-Loop)")
    st.warning("Valide a formatação técnica. As edições aplicadas aqui serão cimentadas no documento Word final.")

    fatos_finais = st.text_area("I. DOS FATOS (Narrativa numerada e objetiva)", value=st.session_state.texto_fatos, height=250)
    direito_final = st.text_area("II. DO DIREITO E JURISPRUDÊNCIA", value=st.session_state.texto_direito, height=250)
    tutela_final = st.text_area("III. DA TUTELA PROVISÓRIA DE URGÊNCIA", value=st.session_state.texto_tutela, height=150)
    pedidos_finais = st.text_area("IV. DOS PEDIDOS (Verbos no infinitivo)", value=st.session_state.texto_pedidos, height=250)

    st.divider()

    # Botão isolado para gerar o arquivo DOCX apenas quando solicitado
    if st.button("💾 Salvar Edições e Gerar Documento Word", type="primary"):
        doc_stream, msg_status = gerar_docx_peticao(fatos_finais, direito_final, tutela_final, pedidos_finais)
        st.session_state.doc_gerado = doc_stream
        st.session_state.msg_status = msg_status

    # Só mostra a área de download se o documento foi gerado com sucesso
    if st.session_state.doc_gerado:
        st.markdown("### 🔔 Checklist de Preenchimento Manual")
        st.info("""
        **Atenção:** O Paralegal Digital redigiu a argumentação jurídica, mas **não possui** os dados sensíveis do cliente.
        Após baixar o arquivo Word, você **precisará** preencher os colchetes restantes:
        - [ ] Qualificação Completa (Nacionalidade, Estado Civil, RG, CPF/CNPJ, Endereço).
        - [ ] Valor da Causa (Numérico e por extenso na Seção V).
        - [ ] Informações do Advogado (Nome, OAB, Data e Local).
        - [ ] Conversão dos links das jurisprudências para hiperlinks nativos do Word.
        """)

        st.success("🎉 Petição formatada com sucesso! O texto injetado preservou os estilos do template original.")
        st.download_button(
            label="📥 Baixar Petição Inicial Final (.docx)",
            data=st.session_state.doc_gerado,
            file_name="Peticao_Inicial_Paralegal.docx",
            mime="application/vnd.openxmlformats-officedocument.wordprocessingml.document",
            use_container_width=True
        )
    elif st.session_state.msg_status:
        st.error(st.session_state.msg_status)

# ─────────────────────────────────────────
# 4. MÓDULO DE AUDITORIA E LOGS (CAIXA PRETA)
# ─────────────────────────────────────────
if st.session_state.etapa >= 2:
    st.divider()
    with st.expander("🛠️ CAIXA PRETA: Auditoria de Sistema e Logs", expanded=False):
        st.markdown("### 🔍 Telemetria Completa do Pipeline V2.2")
        st.info("Utilize estes dados para debugar alucinações, falhas de prompt ou erros na API do Google.")

        log_content = f"""=========================================
AUDITORIA DO SISTEMA - PARALEGAL DIGITAL V2.2
=========================================

[1] TERMOS DE BUSCA ENVIADOS AO GOOGLE:
{st.session_state.get('termos', 'N/A')}

-----------------------------------------
[2] RETORNO BRUTO DO GOOGLE SERPER (RAG):
{st.session_state.get('debug_juris', 'N/A')}

-----------------------------------------
[3] PROMPT ENVIADO AO RELATOR:
{st.session_state.get('debug_prompt', 'N/A')}

-----------------------------------------
[4] RESUMO DOS FATOS (Agente Investigador):
{st.session_state.get('resumo', 'N/A')}

-----------------------------------------
[5] RELATÓRIO DE TRIAGEM (Agente Relator):
{st.session_state.get('relatorio', 'N/A')}

-----------------------------------------
[6] PETIÇÃO INICIAL (Agente Redator):
>> FATOS:
{st.session_state.get('texto_fatos', 'N/A')}

>> DIREITO E JURISPRUDÊNCIA:
{st.session_state.get('texto_direito', 'N/A')}

>> TUTELA PROVISÓRIA:
{st.session_state.get('texto_tutela', 'N/A')}

>> PEDIDOS:
{st.session_state.get('texto_pedidos', 'N/A')}
=========================================
FIM DO LOG
========================================="""

        st.text_area("Visualizador de Logs Internos", value=log_content, height=300)

        st.download_button(
            label="🚨 Baixar Log de Auditoria para Debug (.TXT)",
            data=log_content,
            file_name="auditoria_paralegal_debug.txt",
            mime="text/plain",
            use_container_width=True
        )


Overwriting app_piloto.py


Celula 3 - Ligar com Ngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import os
from huggingface_hub import login

# 1. Resgata os segredos do Colab
os.environ["SERPER_API"] = userdata.get("SERPER_API")

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    # Faz o login oficial na Hugging Face
    login(token=HF_TOKEN)

    # LIGA O MODO TURBO DE DOWNLOAD (hf_transfer)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("✅ Hugging Face: Login efetuado e Modo Turbo ativado!")
except Exception as e:
    print("⚠️ Aviso: HF_TOKEN não encontrado no cofre. O download será na velocidade normal.")

try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok: Token lido com segurança!")
except Exception as e:
    print("❌ ERRO: 'NGROK_TOKEN' não encontrado no menu lateral.")
    raise e

# 2. Limpeza Total e Abertura do Túnel
!pkill -9 -f streamlit
ngrok.kill()

try:
    url_publica = ngrok.connect(8501)
    print("\n" + "="*50)
    print(f"🚀 O SEU PARALEGAL DIGITAL ESTÁ ONLINE!")
    print(f"🔗 ENDEREÇO PARA O SEU IRMÃO: {url_publica.public_url}")
    print("="*50)
    print("\n(Aguarde o carregamento abaixo e mantenha esta célula rodando)\n")

    # 3. Execução do App
    !streamlit run app_piloto.py --server.enableCORS false --server.enableXsrfProtection false

except Exception as e:
    print(f"❌ Erro ao iniciar o Ngrok: {e}")

✅ Hugging Face: Login efetuado e Modo Turbo ativado!
✅ Ngrok: Token lido com segurança!

🚀 O SEU PARALEGAL DIGITAL ESTÁ ONLINE!
🔗 ENDEREÇO PARA O SEU IRMÃO: https://reluctant-film-cognition.ngrok-free.dev

(Aguarde o carregamento abaixo e mantenha esta célula rodando)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.110.62.45:8501

